# Forecasting time series with skforecast-ai

## Introduction

## Libraries

Libraries used in this document.

In [9]:
# Data processing
# ==============================================================================
import numpy as np
import pandas as pd
from skforecast.datasets import fetch_dataset

# Plots
# ==============================================================================
from skforecast.plot import set_dark_theme
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from skforecast.plot import plot_residuals
import plotly.graph_objects as go
import plotly.io as pio
import plotly.offline as poff
pio.templates.default = "seaborn"
poff.init_notebook_mode(connected=True)
plt.style.use('seaborn-v0_8-darkgrid')

# Modelling and Forecasting
# ==============================================================================
import xgboost
import lightgbm
import sklearn
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures
from skforecast.preprocessing import CalendarFeatures
from sklearn.feature_selection import RFECV
from sklearn.compose import make_column_transformer, make_column_selector

import skforecast
from skforecast.recursive import ForecasterEquivalentDate, ForecasterRecursive
from skforecast.model_selection import (
    TimeSeriesFold,
    OneStepAheadFold,
    bayesian_search_forecaster,
    backtesting_forecaster,
)
from skforecast.preprocessing import RollingFeatures
from skforecast.feature_selection import select_features
from skforecast.stats import calculate_lag_autocorrelation
from skforecast.metrics import calculate_coverage

# Warnings configuration
# ==============================================================================
import warnings
warnings.filterwarnings('once')

color = '\033[1m\033[38;5;208m' 
print(f"{color}Version skforecast: {skforecast.__version__}")
print(f"{color}Version scikit-learn: {sklearn.__version__}")
print(f"{color}Version lightgbm: {lightgbm.__version__}")
print(f"{color}Version xgboost: {xgboost.__version__}")
print(f"{color}Version pandas: {pd.__version__}")
print(f"{color}Version numpy: {np.__version__}")

Version skforecast: 0.23.0
Version scikit-learn: 1.8.0
Version lightgbm: 4.6.0
Version xgboost: 3.3.0
Version pandas: 2.3.3
Version numpy: 2.4.6


In [10]:
from skforecast_ai import ForecastingAssistant

## Data

The data in this document represent the hourly usage of the bike share system in the city of Washington, D.C. during the years 2011 and 2012. In addition to the number of users per hour, information about weather conditions and holidays is available. The original data was obtained from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/ml/datasets/bike+sharing+dataset) and has been previously cleaned ([code](https://github.com/JoaquinAmatRodrigo/Estadistica-machine-learning-python/blob/master/code/prepare_bike_sharing_dataset.ipynb)) applying the following modifications:

+ Renamed columns with more descriptive names.

+ Renamed categories of the weather variables. The category of `heavy_rain` has been combined with that of `rain`.

+ Denormalized temperature, humidity and wind variables.

+ `date_time` variable created and set as index.

+ Imputed missing values by forward filling.

In [24]:
# Downloading data
# ==============================================================================
data = fetch_dataset('bike_sharing', raw=True)
data = data[['date_time', 'users', 'holiday', 'weather', 'temp']]
data['date_time'] = pd.to_datetime(data['date_time'], format='%Y-%m-%d %H:%M:%S')
data = data.set_index('date_time')
data = data.asfreq('h')
data = data.sort_index()
data.head()

╭───────────────────────────────── bike_sharing ──────────────────────────────────╮
│ Description:                                                                    │
│ Hourly usage of the bike share system in the city of Washington D.C. during the │
│ years 2011 and 2012. In addition to the number of users per hour, information   │
│ about weather conditions and holidays is available.                             │
│                                                                                 │
│ Source:                                                                         │
│ Fanaee-T,Hadi. (2013). Bike Sharing Dataset. UCI Machine Learning Repository.   │
│ https://doi.org/10.24432/C5W894.                                                │
│                                                                                 │
│ URL:                                                                            │
│ https://raw.githubusercontent.com/skforecast/skforecast-                        │
│ datasets/main/data/bike_sharing_dataset_clean.csv                               │
│                                                                                 │
│ Shape: 17544 rows x 12 columns                                                  │
╰─────────────────────────────────────────────────────────────────────────────────╯

,users,holiday,weather,temp
date_time,,,,
2011-01-01 00:00:00,16.0,0.0,clear,9.84
2011-01-01 01:00:00,40.0,0.0,clear,9.02
2011-01-01 02:00:00,32.0,0.0,clear,9.02
2011-01-01 03:00:00,13.0,0.0,clear,9.84
2011-01-01 04:00:00,1.0,0.0,clear,9.84


In [25]:
assistant = ForecastingAssistant()

In [37]:
profile = assistant.profile(
             data        = data,
             target      = "users",
             date_column = "date_time",
         )

In [38]:
profile.series_pacf

[SeriesPacf(series_id='users', n_observations=17544, lags=[1, 2, 23, 22, 25, 169, 10, 145, 17, 143, 167, 19, 21, 337, 3, 20, 142, 32, 26, 135, 166, 8, 160, 15, 121, 5, 119, 136, 313, 335, 24, 33, 6, 159, 50, 9, 73, 4, 118, 70, 71, 97, 311, 64, 141, 334, 503, 328, 95, 137, 74, 165, 193, 28, 43, 168, 29, 336, 111, 48, 504, 94, 144, 385, 41, 27, 13, 98, 502, 31, 176, 152, 161, 57, 303, 122, 69, 117, 327, 333, 132, 310, 12, 192, 496, 39, 120, 112, 34, 481, 63, 18, 140, 329, 179, 115, 30, 217, 309, 344, 383, 96, 147, 66], pacf_abs=[0.8451720156882964, 0.4081855899587973, 0.35448577123286246, 0.3426733694045754, 0.33189603349204894, 0.3152610898864668, 0.2723148439612434, 0.24365969812949909, 0.24152942731810595, 0.24031390765802554, 0.21751300267914822, 0.19882507861519158, 0.1930441021006387, 0.18414175330542576, 0.1818883564175728, 0.17813353800998744, 0.15492433010109746, 0.15338921889117843, 0.1458000115457937, 0.13605567975003927, 0.13209858090659174, 0.13181017046856922, 0.12050439523

In [26]:
result = assistant.forecast(
             data        = data,
             target      = "users",
             date_column = "date_time",
             steps       = 36,
         )

In [27]:
# Forecasted values for the next 12 steps
print(result.predictions.head(3))

# Evaluation metrics: MAE, MSE, MASE per series
print(result.metrics)

# The complete, standalone Python script that was executed
print(result.code)

                           pred
2012-08-07 19:00:00  662.036933
2012-08-07 20:00:00  484.713662
2012-08-07 21:00:00  333.245872
  series        MAE          MSE      MASE      MAPE
0  users  27.395686  1238.539266  0.461581  0.222602
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
from skforecast.metrics import mean_absolute_scaled_error
from lightgbm import LGBMRegressor
from skforecast.preprocessing import RollingFeatures, CalendarFeatures
from skforecast.recursive import ForecasterRecursive

# Load data
data = pd.read_csv('data.csv', index_col=0, parse_dates=True)

data = data.asfreq('h')
data = data.sort_index()

# Train/test split
end_train = '2012-08-07 18:00:00'  # 80% of data, adjust to change the split point
data_train = data.loc[:end_train]
data_test  = data.loc[data.index > end_train]
exog_features = ['holiday', 'weather', 'temp']

print(
    f"Train dates : {data_train.index.min()} --- {data_train.index

In [28]:
# Split train-validation-test
# ==============================================================================
end_train = '2012-04-30 23:59:00'
end_validation = '2012-08-31 23:59:00'
data_train = data.loc[: end_train, :]
data_val   = data.loc[end_train:end_validation, :]
data_test  = data.loc[end_validation:, :]

print(f"Dates train      : {data_train.index.min()} --- {data_train.index.max()}  (n={len(data_train)})")
print(f"Dates validation : {data_val.index.min()} --- {data_val.index.max()}  (n={len(data_val)})")
print(f"Dates test       : {data_test.index.min()} --- {data_test.index.max()}  (n={len(data_test)})")

Dates train      : 2011-01-01 00:00:00 --- 2012-04-30 23:00:00  (n=11664)
Dates validation : 2012-05-01 00:00:00 --- 2012-08-31 23:00:00  (n=2952)
Dates test       : 2012-09-01 00:00:00 --- 2012-12-31 23:00:00  (n=2928)


In [33]:
cv = TimeSeriesFold(steps = 36, initial_train_size = end_validation)

result = assistant.backtest(
             data        = data,
             target      = "users",
             date_column = "date_time",
             cv          = cv,
         )

Information of folds
--------------------
Number of observations used for initial training: 14616
Number of observations used for backtesting: 2928
    Number of folds: 82
    Number skipped folds: 0 
    Number of steps per fold: 36
    Number of steps to exclude between last observed data (last window) and predictions (gap): 0
    Last fold only includes 12 observations.

Fold: 0
    Training:   2011-01-01 00:00:00 -- 2012-08-31 23:00:00  (n=14616)
    Validation: 2012-09-01 00:00:00 -- 2012-09-02 11:00:00  (n=36)
Fold: 1
    Training:   No training in this fold
    Validation: 2012-09-02 12:00:00 -- 2012-09-03 23:00:00  (n=36)
Fold: 2
    Training:   No training in this fold
    Validation: 2012-09-04 00:00:00 -- 2012-09-05 11:00:00  (n=36)
Fold: 3
    Training:   No training in this fold
    Validation: 2012-09-05 12:00:00 -- 2012-09-06 23:00:00  (n=36)
Fold: 4
    Training:   No training in this fold
    Validation: 2012-09-07 00:00:00 -- 2012-09-08 11:00:00  (n=36)
Fold: 5
    Tr

  0%|          | 0/82 [00:00<?, ?it/s]

In [34]:
print(result.metrics)
print(result.cv_config)     # exactly which folds were run
print(result.explanation)

   mean_absolute_error  mean_squared_error  mean_absolute_scaled_error  \
0            47.222107         5945.024516                     0.75828   

   mean_absolute_percentage_error  
0                        0.568114  
{'steps': 36, 'initial_train_size': '2012-08-31 23:59:00', 'refit': False, 'fixed_train_size': True, 'gap': 0, 'fold_stride': 36, 'differentiation': None}
Initial training up to 2012-08-31 23:59:00, fixed window, no refit, 36-step horizon, 82 folds. Results — mean_absolute_error: 47.2221, mean_squared_error: 5945.0245, mean_absolute_scaled_error: 0.7583, mean_absolute_percentage_error: 0.5681.


In [36]:
print(result.explanation)

Initial training up to 2012-08-31 23:59:00, fixed window, no refit, 36-step horizon, 82 folds. Results — mean_absolute_error: 47.2221, mean_squared_error: 5945.0245, mean_absolute_scaled_error: 0.7583, mean_absolute_percentage_error: 0.5681.


In [35]:
print(result.code)

import pandas as pd
from lightgbm import LGBMRegressor
from skforecast.preprocessing import RollingFeatures, CalendarFeatures
from skforecast.recursive import ForecasterRecursive
from skforecast.model_selection import TimeSeriesFold, backtesting_forecaster

# Load data
data = pd.read_csv('data.csv', index_col=0, parse_dates=True)

data = data.asfreq('h')
data = data.sort_index()

window_features = RollingFeatures(
    stats        = ['mean', 'std', 'mean', 'mean', 'mean'],
    window_sizes = [22, 22, 24, 48, 72],
)

calendar_features = CalendarFeatures(
    features = ['hour', 'day_of_week', 'weekend', 'month'],
    encoding = None,
)

# Create forecaster
forecaster = ForecasterRecursive(
    estimator            = LGBMRegressor(random_state=123, verbose=-1),
    lags                 = [1, 2, 3, 4, 5, 6, 8, 9, 10, 12, 13, 15, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 39, 41, 43, 48, 50, 57, 63, 64, 66, 69, 70, 71, 73, 74, 94, 95, 96, 97, 98, 111, 112, 115,